In [ ]:
pip install azure-storage-blob msal requests python-dotenv

In [ ]:
"""
RIPCON Audiovisual Uploader v2
================================
Mejoras sobre v1:
  - blob.exists() eliminado → listado previo del container (1 op vs N ops)
  - Paralelismo a nivel de proyecto completo, no por semana
  - Semaforo para limitar concurrencia y evitar throttling de Graph
  - Descarga en streaming con chunks para archivos grandes (videos 4K)
  - Estadisticas de costo estimado al finalizar
  - Pool de tokens reutilizable (evita re-autenticar por cada trabajo)

Convencion de nombres:
  Container : audiovisual
  Proyecto  : CODIGO_nombre-corto  (ej. 28002_confluencia-puembo)
  Semana    : YYYY_S##             (ej. 2026_S15)
  Archivo   : PREFIJO_YYYYMMDD_NNN.ext

Requisitos:
  pip install azure-storage-blob msal requests python-dotenv
"""

import os
import re
import sys
import time
import mimetypes
import logging
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv

import requests
from requests.adapters import HTTPAdapter
import msal
from urllib3.util.retry import Retry
from azure.storage.blob import BlobServiceClient, ContentSettings, StandardBlobTier

# ─── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("ripcon")

# ─── Configuracion ───────────────────────────────────────────────────────────
load_dotenv()

def _clean_env(*keys: str) -> str | None:
    for key in keys:
        value = os.getenv(key)
        if value is None:
            continue
        value = value.strip()
        if value and value.lower() not in {"none", "null"}:
            return value
    return None

TENANT_ID        = _clean_env("AZURE_TENANT_ID", "TENANT_ID")
CLIENT_ID        = _clean_env("AZURE_CLIENT_ID", "CLIENT_ID")
CLIENT_SECRET    = _clean_env("AZURE_CLIENT_SECRET", "CLIENT_SECRET")
STORAGE_CONN_STR = _clean_env("AZURE_STORAGE_CONNECTION_STRING", "STORAGE_CONNECTION_STRING")
CONTAINER_NAME   = "audiovisual"

GRAPH_BASE                  = "https://graph.microsoft.com/v1.0"
GRAPH_SCOPE                 = ["https://graph.microsoft.com/.default"]
GRAPH_CONNECT_TIMEOUT       = 15
GRAPH_READ_TIMEOUT          = 120
GRAPH_DOWNLOAD_READ_TIMEOUT = 300
GRAPH_DOWNLOAD_RETRIES      = 3
GRAPH_MAX_RETRIES           = 5
GRAPH_RETRY_BACKOFF         = 2

# Cuanto tiempo (segundos) reutilizar el token antes de renovarlo
TOKEN_TTL_SECONDS = 3000  # tokens duran 3600s, renovamos con margen

# Chunk size para streaming de archivos grandes (32 MB)
CHUNK_SIZE = 32 * 1024 * 1024
GRAPH_DOWNLOAD_CHUNK = 16 * 1024 * 1024

# Limite de conexiones paralelas a Graph API antes de empezar a throttlear
GRAPH_MAX_CONCURRENT = 12
GRAPH_HTTP_POOL_SIZE = 32

# ─── Extensiones ────────────────────────────────────────────────────────────
EXT_PREFIX = {
    ".dng":  "DRN", ".cr3": "DRN", ".arw": "DRN",
    ".insv": "I360",
    "jpeg": "FOT",
    ".mp4":  "VID", ".mov": "VID", ".mkv": "VID",
    ".avi":  "VID", ".mts": "VID", ".m2ts": "VID",
    ".srt":  "VID", ".lrf": "VID",
    ".tiff": "FOT", ".tif": "FOT",
    ".png":  "FOT", ".heic": "FOT", ".heif": "FOT",
}
OFFICE_EXTS = {
    ".doc", ".docx", ".xls", ".xlsx",
    ".ppt", ".pptx", ".odt", ".ods",
    ".odp", ".rtf", ".pdf", ".txt",
    ".rar", "zip", ".7z", "7zip",
}
SKIP_EXTENSIONS = OFFICE_EXTS


def validar_configuracion() -> None:
    faltantes = [
        k for k, v in {
            "AZURE_TENANT_ID": TENANT_ID,
            "AZURE_CLIENT_ID": CLIENT_ID,
            "AZURE_CLIENT_SECRET": CLIENT_SECRET,
            "AZURE_STORAGE_CONNECTION_STRING": STORAGE_CONN_STR,
        }.items() if not v
    ]
    if faltantes:
        raise RuntimeError(
            f"Faltan variables de entorno: {', '.join(faltantes)}\n"
            "Crea un archivo .env con esas variables."
        )


# ════════════════════════════════════════════════════════════════════════════
# 1. TOKEN POOL — reutiliza token, renueva solo cuando expira
# ════════════════════════════════════════════════════════════════════════════

class TokenPool:
    """Thread-safe cache de token para Graph API."""

    def __init__(self) -> None:
        self._token: str | None = None
        self._expira: float = 0.0
        self._lock = threading.Lock()
        self._app: msal.ConfidentialClientApplication | None = None

    def _build_app(self) -> msal.ConfidentialClientApplication:
        if self._app is None:
            validar_configuracion()
            self._app = msal.ConfidentialClientApplication(
                CLIENT_ID,
                authority=f"https://login.microsoftonline.com/{TENANT_ID}",
                client_credential=CLIENT_SECRET,
            )
        return self._app

    def get(self, force_refresh: bool = False) -> str:
        with self._lock:
            if (not force_refresh) and self._token and time.monotonic() < self._expira:
                return self._token
            app = self._build_app()
            result = app.acquire_token_for_client(scopes=GRAPH_SCOPE)
            if "access_token" not in result:
                raise RuntimeError(
                    f"Error autenticacion Graph: {result.get('error_description')}"
                )
            self._token = result["access_token"]
            expires_in = int(result.get("expires_in", TOKEN_TTL_SECONDS))
            self._expira = time.monotonic() + max(60, min(TOKEN_TTL_SECONDS, expires_in - 120))
            log.info("Token Graph renovado.")
            return self._token


# Instancia global — todos los hilos la comparten
_token_pool = TokenPool()


def get_graph_token(force_refresh: bool = False) -> str:
    return _token_pool.get(force_refresh=force_refresh)


# ════════════════════════════════════════════════════════════════════════════
# 2. HTTP helpers con reintentos
# ════════════════════════════════════════════════════════════════════════════

def _with_retry(fn, max_retries: int = GRAPH_MAX_RETRIES) -> any:
    """Ejecuta fn() con reintentos exponenciales para errores transitorios."""
    last_error = None
    for intento in range(1, max_retries + 1):
        try:
            resp = fn()
            if hasattr(resp, "status_code") and resp.status_code in {429, 500, 502, 503, 504}:
                if intento == max_retries:
                    resp.raise_for_status()
                retry_after = resp.headers.get("Retry-After")
                espera = (
                    int(retry_after)
                    if retry_after and retry_after.isdigit()
                    else GRAPH_RETRY_BACKOFF ** intento
                )
                log.warning(
                    f"HTTP {resp.status_code}. Reintento {intento}/{max_retries} en {espera}s"
                )
                time.sleep(espera)
                continue
            return resp
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            last_error = e
            if intento == max_retries:
                break
            espera = GRAPH_RETRY_BACKOFF ** intento
            log.warning(
                f"Timeout/conexion. Reintento {intento}/{max_retries} en {espera}s"
            )
            time.sleep(espera)
    raise RuntimeError(f"Fallo tras {max_retries} intentos: {last_error}")


_http_local = threading.local()


def _get_http_session() -> requests.Session:
    session = getattr(_http_local, "session", None)
    if session is None:
        session = requests.Session()
        adapter = HTTPAdapter(pool_connections=GRAPH_HTTP_POOL_SIZE, pool_maxsize=GRAPH_HTTP_POOL_SIZE)
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        _http_local.session = session
    return session


def graph_get(token: str, url: str) -> dict:
    def _call():
        return _get_http_session().get(
            url,
            headers={"Authorization": f"Bearer {token}"},
            timeout=(GRAPH_CONNECT_TIMEOUT, GRAPH_READ_TIMEOUT),
        )

    resp = _with_retry(_call)
    resp.raise_for_status()
    return resp.json()


def graph_download_stream(token: str, download_url: str) -> bytes:
    """
    Descarga en chunks para no cargar archivos grandes completos en RAM de golpe.
    Para un video de 2GB descarga 16MB a la vez y los concatena.
    """
    def _call():
        return _get_http_session().get(
            download_url,
            headers={"Authorization": f"Bearer {token}"},
            timeout=(GRAPH_CONNECT_TIMEOUT, GRAPH_DOWNLOAD_READ_TIMEOUT),
            stream=True,
        )

    for intento in range(1, GRAPH_DOWNLOAD_RETRIES + 1):
        try:
            resp = _with_retry(_call)
            resp.raise_for_status()

            chunks = []
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    chunks.append(chunk)
            return b"".join(chunks)
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.Timeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError) as e:
            if intento == GRAPH_DOWNLOAD_RETRIES:
                raise
            espera = GRAPH_RETRY_BACKOFF ** intento
            log.warning(
                f"Read timeout/conexion en descarga {download_url}. "
                f"Reintento {intento}/{GRAPH_DOWNLOAD_RETRIES} en {espera}s"
            )
            time.sleep(espera)

    raise RuntimeError(f"Fallo tras {GRAPH_DOWNLOAD_RETRIES} intentos de descarga: {download_url}")


def graph_list_all(token: str, url: str) -> list[dict]:
    """Lista todos los elementos manejando paginacion."""
    elementos: list[dict] = []
    next_url: str | None = url
    while next_url:
        data = graph_get(token, next_url)
        elementos.extend(data.get("value", []))
        next_url = data.get("@odata.nextLink")
    return elementos


# ════════════════════════════════════════════════════════════════════════════
# 3. LISTAR ARCHIVOS
# ════════════════════════════════════════════════════════════════════


def _url_children(base_drive_url: str, folder_path: str) -> str:
    normalized = requests.utils.unquote(folder_path or "/")
    if normalized == "/":
        return f"{base_drive_url}/root/children"
    encoded = requests.utils.quote(normalized, safe="/")
    return f"{base_drive_url}/root:{encoded}:/children"


def _listar_archivos_drive(
    token: str,
    base_drive_url: str,
    folder_path: str = "/",
    recursive: bool = True,
) -> tuple[list[dict], int]:
    pendientes = [_url_children(base_drive_url, folder_path)]
    files: list[dict] = []
    carpetas = 0

    while pendientes:
        items = graph_list_all(token, pendientes.pop(0))
        for item in items:
            if "file" in item:
                files.append(item)
            elif recursive and "folder" in item:
                drive_id = item.get("parentReference", {}).get("driveId")
                item_id = item.get("id")
                if drive_id and item_id:
                    pendientes.append(
                        f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/children"
                    )
                    carpetas += 1
    return files, carpetas


def list_sharepoint_files(
    token: str, site_id: str, folder_path: str = "/", recursive: bool = True
) -> list[dict]:
    base = f"{GRAPH_BASE}/sites/{site_id}/drive"
    files, carpetas = _listar_archivos_drive(token, base, folder_path, recursive)
    log.info(
        f"SharePoint: {len(files)} archivos | {carpetas} subcarpetas en {folder_path}"
    )
    return files


def list_onedrive_files(
    token: str, user_email: str, folder_path: str = "/", recursive: bool = True
) -> list[dict]:
    base = f"{GRAPH_BASE}/users/{user_email}/drive"
    files, carpetas = _listar_archivos_drive(token, base, folder_path, recursive)
    log.info(
        f"OneDrive ({user_email}): {len(files)} archivos | {carpetas} subcarpetas en {folder_path}"
    )
    return files


# ════════════════════════════════════════════════════════════════════════════
# 4. BLOB INDEX — lista el container UNA SOLA VEZ
#    Esto reemplaza el blob.exists() individual (N llamadas → 1 llamada)
# ════════════════════════════════════════════════════════════════════════════

def construir_blob_index(blob_svc: BlobServiceClient, prefix: str = "") -> set[str]:
    """
    Lista todos los blobs del container con el prefijo dado y retorna un set de rutas.

    En lugar de llamar blob.exists() para cada archivo (N operaciones de lectura),
    hacemos UNA SOLA llamada list_blobs y guardamos todo en memoria.

    Costo: 1 operacion de lista vs N operaciones de lectura individual.

    Para 2000 archivos:
      - blob.exists() x2000 = 2000 ops = ~$0.002
      - list_blobs x1       =    1 op  = ~$0.000001
    """
    container = blob_svc.get_container_client(CONTAINER_NAME)
    blobs = container.list_blobs(name_starts_with=prefix)
    index = {blob.name for blob in blobs}
    log.info(f"Blob index construido: {len(index)} blobs existentes (prefijo: '{prefix or '/'}')")
    return index


# ════════════════════════════════════════════════════════════════════════════
# 5. RENOMBRADO
# ════════════════════════════════════════════════════════════════════════════


def ruta_humana_item(item: dict) -> str:
    raw_path = item.get("parentReference", {}).get("path", "")
    if "root:" in raw_path:
        return raw_path.split("root:", 1)[1]
    return raw_path


def detectar_prefijo(nombre_original: str, prefijo_jpg: str = "FOT") -> str:
    ext = Path(nombre_original).suffix.lower()
    if ext in {".jpg", ".jpeg", ".heic", ".heif"}:
        return prefijo_jpg
    if ext in OFFICE_EXTS:
        return "SKIP"
    return EXT_PREFIX.get(ext, "FOT")


def numero_semana_iso(fecha: datetime) -> str:
    year, week, _ = fecha.isocalendar()
    return f"{year}_S{week:02d}"


def renombrar_archivo(
    nombre_original: str,
    fecha_captura: datetime,
    secuencia: int,
    prefijo_jpg: str = "FOT",
) -> str:
    prefijo = detectar_prefijo(nombre_original, prefijo_jpg)
    fecha_str = fecha_captura.strftime("%Y%m%d")
    ext = Path(nombre_original).suffix.lower()
    return f"{prefijo}_{fecha_str}_{secuencia:03d}{ext}"


def extraer_fecha_desde_texto(texto: str) -> datetime | None:
    if not texto:
        return None
    # Prioridad: timestamps tipo YYYYMMDDHHMMSS -> tomar los primeros 8 digitos.
    m_ts = re.search(r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])([0-2]\d|3[0-1])\d{6}(?!\d)", texto)
    if m_ts:
        try:
            y, mo, d = int(m_ts.group(1)), int(m_ts.group(2)), int(m_ts.group(3))
            return datetime(y, mo, d, tzinfo=timezone.utc)
        except ValueError:
            pass
    patrones = [
        ("ymd4", r"(?<!\d)(20\d{2})[-_./]?(0[1-9]|1[0-2])[-_./]?([0-2]\d|3[0-1])(?!\d)"),
        ("dmy4", r"(?<!\d)([0-2]\d|3[0-1])[-_./](0[1-9]|1[0-2])[-_./](20\d{2})(?!\d)"),
        ("ymd2", r"(?<!\d)(\d{2})(0[1-9]|1[0-2])([0-2]\d|3[0-1])(?!\d)"),
    ]
    for tipo, patron in patrones:
        for m in re.finditer(patron, texto):
            try:
                if tipo == "ymd4":
                    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
                elif tipo == "dmy4":
                    d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
                else:
                    y, mo, d = 2000 + int(m.group(1)), int(m.group(2)), int(m.group(3))
                return datetime(y, mo, d, tzinfo=timezone.utc)
            except ValueError:
                continue
    return None


def extraer_semana_desde_texto(texto: str) -> datetime | None:
    if not texto:
        return None
    m = re.search(r"(20\d{2})\s*[_-]?\s*[sS](\d{1,2})", texto)
    if m:
        year, week = int(m.group(1)), int(m.group(2))
        if 1 <= week <= 53:
            return datetime.fromisocalendar(year, week, 1).replace(tzinfo=timezone.utc)
    m2 = re.search(r"semana\s*[_-]?(\d{1,2})", texto, flags=re.IGNORECASE)
    m_year = re.search(r"(20\d{2})", texto)
    if m2 and m_year:
        week, year = int(m2.group(1)), int(m_year.group(1))
        if 1 <= week <= 53:
            return datetime.fromisocalendar(year, week, 1).replace(tzinfo=timezone.utc)
    return None


def inferir_fecha(item: dict) -> datetime:
    nombre = item.get("name", "")
    ruta = ruta_humana_item(item)

    # Priorizar siempre la fecha embebida en el nombre del archivo.
    fecha = extraer_fecha_desde_texto(nombre) or extraer_semana_desde_texto(nombre)

    # Si el nombre no trae una fecha utilizable, probar la ruta completa.
    if not fecha:
        texto = " | ".join(filter(None, [nombre, ruta]))
        fecha = extraer_fecha_desde_texto(texto) or extraer_semana_desde_texto(texto)
    if fecha:
        return fecha
    file_info = item.get("fileSystemInfo", {})
    created = file_info.get("createdDateTime") or item.get("lastModifiedDateTime")
    if created:
        return datetime.fromisoformat(created.replace("Z", "+00:00"))
    return datetime.now(timezone.utc)


# ════════════════════════════════════════════════════════════════════════════
# 6. SUBIDA A BLOB
# ════════════════════════════════════════════════════════════════════════════

def get_blob_client() -> BlobServiceClient:
    return BlobServiceClient.from_connection_string(STORAGE_CONN_STR)


def content_type_para(nombre: str) -> str:
    ext = Path(nombre).suffix.lower()
    tipos = {
        ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".tiff": "image/tiff", ".tif": "image/tiff",
        ".heic": "image/heic", ".heif": "image/heif",
        ".dng": "image/x-adobe-dng",
        ".cr3": "image/x-canon-cr3",
        ".arw": "image/x-sony-arw",
        ".mp4": "video/mp4",
        ".mov": "video/quicktime",
        ".mkv": "video/x-matroska",
        ".avi": "video/x-msvideo",
        ".mts": "video/mp2t", ".m2ts": "video/mp2t",
        ".srt": "text/plain; charset=utf-8",
        ".insv": "application/octet-stream",
        ".lrf": "application/octet-stream",
    }
    tipo = tipos.get(ext)
    if tipo:
        return tipo
    guess, _ = mimetypes.guess_type(nombre)
    return guess or "application/octet-stream"


def subir_a_blob(
    blob_svc: BlobServiceClient,
    contenido: bytes,
    blob_path: str,
    content_type: str = "application/octet-stream",
    tier: StandardBlobTier = StandardBlobTier.COOL,
) -> None:
    container = blob_svc.get_container_client(CONTAINER_NAME)
    container.upload_blob(
        name=blob_path,
        data=contenido,
        overwrite=False,
        standard_blob_tier=tier,
        content_settings=ContentSettings(content_type=content_type),
    )


# ════════════════════════════════════════════════════════════════════════════
# 7. PROCESAMIENTO PARALELO
# ════════════════════════════════════════════════════════════════════════════

def _preparar_trabajos(
    archivos: list[dict],
    carpeta_proyecto: str,
    prefijo_jpg: str,
) -> list[dict]:
    """
    Calcula secuencias y rutas destino para todos los archivos.
    Retorna lista de trabajos listos para ejecutar en paralelo.
    """
    contadores: dict[str, int] = {}
    trabajos: list[dict] = []

    archivos_ordenados = sorted(archivos, key=inferir_fecha)
    for item in archivos_ordenados:
        nombre_orig = item.get("name", "")
        ext = Path(nombre_orig).suffix.lower()

        if ext in SKIP_EXTENSIONS:
            continue

        fecha = inferir_fecha(item)
        semana = numero_semana_iso(fecha)
        prefijo = detectar_prefijo(nombre_orig, prefijo_jpg)

        if prefijo == "SKIP":
            continue

        clave = f"{prefijo}_{fecha.strftime('%Y%m%d')}"
        contadores[clave] = contadores.get(clave, 0) + 1
        seq = contadores[clave]

        nombre_nuevo = renombrar_archivo(nombre_orig, fecha, seq, prefijo_jpg)
        blob_path = f"{carpeta_proyecto}/{semana}/{nombre_nuevo}"

        drive_id = item.get("parentReference", {}).get("driveId")
        item_id = item.get("id")
        if drive_id and item_id:
            dl_url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/content"
        else:
            dl_url = item.get("@microsoft.graph.downloadUrl")

        if not dl_url:
            log.warning(f"No se pudo determinar URL de descarga para {nombre_orig}")
            continue

        trabajos.append({
            "nombre_orig": nombre_orig,
            "nombre_nuevo": nombre_nuevo,
            "blob_path": blob_path,
            "dl_url": dl_url,
            "size_bytes": item.get("size", 0),
        })

    return trabajos
def procesar_archivos(
    archivos: list[dict],
    blob_svc: BlobServiceClient,
    codigo_proyecto: str,
    nombre_corto: str,
    prefijo_jpg: str = "FOT",
    max_workers: int = 4,
) -> dict:
    """
    Procesa todos los archivos de un proyecto con las siguientes optimizaciones:

    1. Construye el blob index UNA vez (1 op de lista vs N ops de lectura)
    2. Paralela descarga+subida con semaforo para no saturar Graph API
    3. Streaming de chunks para archivos grandes
    """
    carpeta_proyecto = f"{codigo_proyecto}_{nombre_corto}"
    resumen = {
        "subidos": 0,
        "omitidos": 0,
        "errores": 0,
        "bytes_subidos": 0,
    }

    trabajos = _preparar_trabajos(archivos, carpeta_proyecto, prefijo_jpg)
    if not trabajos:
        log.info(f"Sin archivos para procesar en {carpeta_proyecto}")
        return resumen

    # ── Paso 1: construir index de blobs existentes (1 operacion) ──────────
    blob_index = construir_blob_index(blob_svc, prefix=carpeta_proyecto)

    # ── Paso 2: separar nuevos vs ya existentes ─────────────────────────────
    nuevos = [t for t in trabajos if t["blob_path"] not in blob_index]
    ya_existentes = [t for t in trabajos if t["blob_path"] in blob_index]

    resumen["omitidos"] = len(ya_existentes)
    if ya_existentes:
        log.info(f"  ⏭️  {len(ya_existentes)} archivos ya existen en Blob, omitidos.")

    if not nuevos:
        log.info(f"  ✅  Nada nuevo que subir en {carpeta_proyecto}")
        return resumen

    log.info(f"📁  {carpeta_proyecto}: {len(nuevos)} archivos nuevos a subir")

    semaforo = threading.Semaphore(GRAPH_MAX_CONCURRENT)
    token = get_graph_token()

    def _subir_uno(trabajo: dict) -> tuple[str, int | str]:
        with semaforo:
            try:
                log.info(
                    f"  📄  {trabajo['nombre_orig']}  →  {trabajo['nombre_nuevo']}"
                    f"  ({trabajo['size_bytes'] / 1024 / 1024:.1f} MB)"
                )
                token_actual = token
                try:
                    contenido = graph_download_stream(token_actual, trabajo["dl_url"])
                except requests.exceptions.HTTPError as e:
                    if "401" in str(e) and "graph.microsoft.com" in trabajo.get("dl_url", ""):
                        log.warning(
                            f"401 en descarga Graph. Reintentando con token renovado: {trabajo['nombre_orig']}"
                        )
                        token_actual = get_graph_token(force_refresh=True)
                        contenido = graph_download_stream(token_actual, trabajo["dl_url"])
                    else:
                        raise

                subir_a_blob(
                    blob_svc,
                    contenido,
                    trabajo["blob_path"],
                    content_type=content_type_para(trabajo["nombre_nuevo"]),
                )
                return "ok", trabajo["size_bytes"]
            except Exception as e:
                return "error", str(e)

    workers = max(1, min(max_workers, len(nuevos)))
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="ripcon") as executor:
        futures = {executor.submit(_subir_uno, t): t for t in nuevos}
        for future in as_completed(futures):
            estado, payload = future.result()
            if estado == "ok":
                resumen["subidos"] += 1
                resumen["bytes_subidos"] += payload or 0
                log.info(f"  ✅  Subido: {futures[future]['blob_path']}")
            else:
                resumen["errores"] += 1
                log.error(
                    f"  ❌  Error con {futures[future]['nombre_orig']}: {payload}"
                )

    return resumen
def estimar_costo(resumen_total: dict) -> None:
    """
    Muestra estimacion de costo de Azure al finalizar.

    Precios Cool tier (East US / Central US aprox):
      Almacenamiento : $0.015 / GB / mes
      Escritura      : $0.10  / 10,000 operaciones
      Lectura        : $0.01  / 10,000 operaciones
      Egress         : $0.087 / GB (primeros 10 TB)
    """
    gb_subidos = resumen_total.get("bytes_subidos", 0) / (1024 ** 3)
    ops_escritura = resumen_total.get("subidos", 0)
    ops_lectura_index = resumen_total.get("index_calls", 1)  # 1 por proyecto

    costo_almacenamiento = gb_subidos * 0.015
    costo_escritura = (ops_escritura / 10_000) * 0.10
    costo_lectura = (ops_lectura_index / 10_000) * 0.01
    costo_egress = gb_subidos * 0.087  # descarga futura estimada ~100% del subido
    log.info(f"     (Si descargas todo  : +${costo_egress:.4f} de egress)")
    costo_total = costo_almacenamiento + costo_escritura + costo_lectura

    log.info("\n💰  Estimacion de costo de esta sesion (Cool tier):")
    log.info(f"     GB subidos          : {gb_subidos:.2f} GB")
    log.info(f"     Almacenamiento/mes  : ${costo_almacenamiento:.4f}")
    log.info(f"     Ops escritura       : ${costo_escritura:.4f}")
    log.info(f"     Ops lectura (index) : ${costo_lectura:.6f}")
    log.info(f"     ─────────────────────────────")
    log.info(f"     TOTAL sesion        : ${costo_total:.4f}")
    log.info(f"     (Si descargas todo  : +${costo_egress:.4f} de egress)")


# ════════════════════════════════════════════════════════════════════════════
# 9. FUNCIONES DE ALTO NIVEL
# ════════════════════════════════════════════════════════════════════════════

def upload_from_sharepoint(
    site_id: str,
    folder_path: str,
    codigo_proyecto: str,
    nombre_corto: str,
    prefijo_jpg: str = "FOT",
    recursive: bool = True,
    max_workers: int = 4,
) -> dict:
    """
    Extrae archivos de SharePoint y los sube a Blob Storage.

    Ejemplo:
        upload_from_sharepoint(
            site_id="ripcon.sharepoint.com,abc123,def456",
            folder_path="/Proyectos/28002/Semana15",
            codigo_proyecto="28002",
            nombre_corto="confluencia-puembo",
            prefijo_jpg="FOT",
        )
    """
    log.info(f"\n🚀  SharePoint → {codigo_proyecto}_{nombre_corto}")
    token = get_graph_token()
    archivos = list_sharepoint_files(token, site_id, folder_path, recursive)
    blob_svc = get_blob_client()
    resumen = procesar_archivos(
        archivos, blob_svc, codigo_proyecto, nombre_corto, prefijo_jpg, max_workers
    )
    log.info(
        f"\n📊  {codigo_proyecto}: "
        f"{resumen['subidos']} subidos | "
        f"{resumen['omitidos']} omitidos | "
        f"{resumen['errores']} errores"
    )
    return resumen


def upload_from_onedrive(
    user_email: str,
    folder_path: str,
    codigo_proyecto: str,
    nombre_corto: str,
    prefijo_jpg: str = "FOT",
    recursive: bool = True,
    max_workers: int = 4,
) -> dict:
    """
    Extrae archivos de OneDrive de un usuario y los sube a Blob Storage.

    Ejemplo:
        upload_from_onedrive(
            user_email="juan@ripcon.com",
            folder_path="/Fotos Obra/Semana 15",
            codigo_proyecto="33014",
            nombre_corto="promart-quitumbe",
            prefijo_jpg="DRN",
        )
    """
    log.info(f"\n🚀  OneDrive ({user_email}) → {codigo_proyecto}_{nombre_corto}")
    token = get_graph_token()
    archivos = list_onedrive_files(token, user_email, folder_path, recursive)
    blob_svc = get_blob_client()
    resumen = procesar_archivos(
        archivos, blob_svc, codigo_proyecto, nombre_corto, prefijo_jpg, max_workers
    )
    log.info(
        f"\n📊  {codigo_proyecto}: "
        f"{resumen['subidos']} subidos | "
        f"{resumen['omitidos']} omitidos | "
        f"{resumen['errores']} errores"
    )
    return resumen


def resolver_sharepoint_desde_url(url: str) -> dict:
    """
    Parsea una URL de SharePoint y retorna site_id, folder_path, etc.

    Ejemplo URL:
        https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/
        33007%20DISTRIBUIDOR%20DE%20TRAFICO%20MONAY/03.%20INFORMACION%20TECNICA%20MONAY/
        03.10%20FOTOS%20Y%20VIDEOS

    Retorna:
        {
            'host': 'ripconcivcialtda.sharepoint.com',
            'site_rel_path': 'sites/GestindeProyectos',
            'site_name': 'GestindeProyectos',
            'site_id': 'SITE_ID_COMPLETO',
            'folder_path': '/33007.../03.10...',
        }
    """
    from urllib.parse import urlparse, unquote

    parsed = urlparse(url)
    host = parsed.netloc
    path_parts = [p for p in parsed.path.split("/") if p]
    if len(path_parts) < 3 or path_parts[0] != "sites":
        raise ValueError(f"URL no tiene estructura /sites/SITIO: {url}")

    site_name = path_parts[1]
    site_rel_path = f"sites/{site_name}"
    library_name = unquote(path_parts[2]) if len(path_parts) > 2 else "Documentos compartidos"
    folder_path_raw = path_parts[3:] if len(path_parts) > 3 else []
    folder_path = "/" + "/".join(unquote(p) for p in folder_path_raw)

    log.info(f"SharePoint URL Parser: {host} | {site_rel_path} | {library_name}")
    token = get_graph_token()

    try:
        site_data = graph_get(token, f"{GRAPH_BASE}/sites/{host}:/{site_rel_path}")
        site_id = site_data.get("id")
    except Exception as e:
        log.error(f"Error obteniendo site_id: {e}")
        site_id = None

    return {
        "host": host,
        "site_rel_path": site_rel_path,
        "site_name": site_name,
        "site_id": site_id,
        "folder_path": folder_path,
    }


JPEG_AS_FOT_WHEN_DRN = True


def detectar_prefijo(nombre_original: str, prefijo_jpg: str = "FOT") -> str:
    """
    Regla normal:
      - .jpg/.jpeg/.heic/.heif usan prefijo_jpg

    Regla especial:
      - si prefijo_jpg = DRN y archivo es .jpeg => usar FOT
        (para casos donde .jpg es dron pero .jpeg es foto normal)
    """
    ext = Path(nombre_original).suffix.lower()
    pref = (prefijo_jpg or "FOT").strip().upper()

    if ext == ".jpg":
        return pref
    if ext == ".jpeg":
        if JPEG_AS_FOT_WHEN_DRN and pref == "DRN":
            return "FOT"
        return pref
    if ext in {".heic", ".heif"}:
        return pref
    if ext in OFFICE_EXTS:
        return "SKIP"
    return EXT_PREFIX.get(ext, "FOT")


def _semana_iso_segura(year: int, week: int) -> datetime | None:
    if not (1 <= week <= 53):
        return None
    max_week = datetime(year, 12, 28).isocalendar().week
    week_ajustada = min(week, max_week)
    return datetime.fromisocalendar(year, week_ajustada, 1).replace(tzinfo=timezone.utc)


def extraer_semana_desde_texto(texto: str) -> datetime | None:
    if not texto:
        return None

    m = re.search(r"(20\d{2})\s*[_-]?\s*[sS](\d{1,2})", texto)
    if m:
        year, week = int(m.group(1)), int(m.group(2))
        fecha = _semana_iso_segura(year, week)
        if fecha:
            return fecha

    m2 = re.search(r"semana\s*[_-]?(\d{1,2})", texto, flags=re.IGNORECASE)
    m_year = re.search(r"(20\d{2})", texto)
    if m2 and m_year:
        week, year = int(m2.group(1)), int(m_year.group(1))
        return _semana_iso_segura(year, week)

    return None


In [ ]:
import tempfile
import subprocess
import shutil

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".mts", ".m2ts", ".lrf", ".insv"}
VIDEO_HEAVY_THRESHOLD_BYTES = 500 * 1024 * 1024
DOWNLOAD_CHUNK_SMALL = 8 * 1024 * 1024
VIDEO_UPLOAD_CONCURRENCY = 8
LIGHT_UPLOAD_CONCURRENCY = 4
HEAVY_MAX_WORKERS = 16

# ── Faststart ────────────────────────────────────────────────────────────────
# Mueve el indice (atomo 'moov') al INICIO del MP4/MOV para que el hub lo
# reproduzca al instante sin bajar todo el archivo. Es un remux SIN recodificar
# (sin perdida de calidad, rapido: solo reordena). Requiere ffmpeg en el PATH.
FFMPEG_BIN = shutil.which("ffmpeg")
FASTSTART_ENABLED = FFMPEG_BIN is not None
FASTSTART_EXTS = {".mp4", ".m4v", ".mov"}
if not FASTSTART_ENABLED:
    log.warning(
        "ffmpeg NO encontrado en el PATH: los videos se subiran SIN faststart "
        "(arrancaran lento en el hub). Instala ffmpeg (https://ffmpeg.org/download.html) "
        "y vuelve a ejecutar esta celda para activarlo."
    )
else:
    log.info(f"Faststart activo (ffmpeg: {FFMPEG_BIN}). Los MP4/MOV se optimizaran al subir.")


def es_video_archivo(nombre: str) -> bool:
    return Path(nombre).suffix.lower() in VIDEO_EXTS


def _es_mp4_mov(nombre: str) -> bool:
    return Path(nombre).suffix.lower() in FASTSTART_EXTS


def _mp4_faststart_state(path: str):
    """True si 'moov' va antes que 'mdat' (ya optimizado); False si al reves
    (necesita faststart); None si no se pudo determinar."""
    try:
        with open(path, "rb") as f:
            pos = 0
            for _ in range(32):
                f.seek(pos)
                header = f.read(16)
                if len(header) < 8:
                    return None
                size = int.from_bytes(header[:4], "big")
                box = header[4:8]
                if box == b"moov":
                    return True
                if box == b"mdat":
                    return False
                if size == 1:            # tamano de 64 bits
                    size = int.from_bytes(header[8:16], "big")
                elif size == 0:          # el box llega hasta el final
                    return None
                if size <= 0:
                    return None
                pos += size
    except OSError:
        return None
    return None


def _remux_faststart(src_path: str, suffix: str) -> str:
    """Reescribe el video con el indice al frente. Copia streams (sin recodificar)."""
    dst_path = src_path + ".fs" + (suffix or ".mp4")
    subprocess.run(
        [FFMPEG_BIN, "-y", "-loglevel", "error", "-i", src_path,
         "-map", "0", "-c", "copy", "-movflags", "+faststart", dst_path],
        check=True,
    )
    return dst_path


def es_trabajo_pesado(trabajo: dict) -> bool:
    size_bytes = int(trabajo.get("size_bytes", 0) or 0)
    nombre = trabajo.get("nombre_nuevo") or trabajo.get("nombre_orig") or ""
    return es_video_archivo(nombre) or size_bytes >= VIDEO_HEAVY_THRESHOLD_BYTES


def graph_download_to_path(token: str, download_url: str, destino_path: str, chunk_size: int = CHUNK_SIZE) -> str:
    def _call():
        return _get_http_session().get(
            download_url,
            headers={"Authorization": f"Bearer {token}"},
            timeout=(GRAPH_CONNECT_TIMEOUT, GRAPH_DOWNLOAD_READ_TIMEOUT),
            stream=True,
        )

    for intento in range(1, GRAPH_DOWNLOAD_RETRIES + 1):
        try:
            resp = _with_retry(_call)
            resp.raise_for_status()

            with open(destino_path, "wb") as salida:
                for chunk in resp.iter_content(chunk_size=chunk_size):
                    if chunk:
                        salida.write(chunk)

            return destino_path
        except (requests.exceptions.ReadTimeout, requests.exceptions.Timeout, requests.exceptions.ConnectionError, requests.exceptions.ChunkedEncodingError) as e:
            if intento == GRAPH_DOWNLOAD_RETRIES:
                raise
            espera = GRAPH_RETRY_BACKOFF ** intento
            log.warning(
                f"Read timeout/conexion en descarga {download_url}. Reintento {intento}/{GRAPH_DOWNLOAD_RETRIES} en {espera}s"
            )
            time.sleep(espera)
            continue


def subir_a_blob_desde_path(
    blob_svc: BlobServiceClient,
    local_path: str,
    blob_path: str,
    content_type: str = "application/octet-stream",
    tier: StandardBlobTier = StandardBlobTier.COOL,
    max_concurrency: int = 8,
) -> None:
    container = blob_svc.get_container_client(CONTAINER_NAME)
    with open(local_path, "rb") as contenido:
        container.upload_blob(
            name=blob_path,
            data=contenido,
            overwrite=False,
            standard_blob_tier=tier,
            max_concurrency=max_concurrency,
            content_settings=ContentSettings(content_type=content_type),
        )


def _procesar_un_trabajo(
    trabajo: dict,
    blob_svc: BlobServiceClient,
    semaforo: threading.Semaphore,
    token_base: str,
    chunk_size: int,
    upload_concurrency: int,
) -> tuple[str, dict | str]:
    with semaforo:
        tmp_path = None
        remux_path = None
        try:
            log.info(
                f"  📄  {trabajo['nombre_orig']}  →  {trabajo['nombre_nuevo']}"
                f"  ({trabajo['size_bytes'] / 1024 / 1024:.1f} MB)"
            )
            t0 = time.perf_counter()
            suffix = Path(trabajo["nombre_nuevo"]).suffix or ".bin"
            with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp_file:
                tmp_path = tmp_file.name

            token_actual = token_base
            try:
                t_dl_ini = time.perf_counter()
                graph_download_to_path(token_actual, trabajo["dl_url"], tmp_path, chunk_size=chunk_size)
                t_dl = time.perf_counter() - t_dl_ini
            except requests.exceptions.HTTPError as e:
                if "401" in str(e) and "graph.microsoft.com" in trabajo.get("dl_url", ""):
                    log.warning(f"401 en descarga Graph. Reintentando con token renovado: {trabajo['nombre_orig']}")
                    token_actual = get_graph_token(force_refresh=True)
                    t_dl_ini = time.perf_counter()
                    graph_download_to_path(token_actual, trabajo["dl_url"], tmp_path, chunk_size=chunk_size)
                    t_dl = time.perf_counter() - t_dl_ini
                else:
                    raise

            # Faststart: si es MP4/MOV con el indice al final, lo reescribimos al
            # frente antes de subir. Si ffmpeg falla, subimos el original igual.
            upload_path = tmp_path
            if FASTSTART_ENABLED and _es_mp4_mov(trabajo["nombre_nuevo"]):
                try:
                    if _mp4_faststart_state(tmp_path) is False:
                        remux_path = _remux_faststart(tmp_path, suffix)
                        upload_path = remux_path
                        log.info(f"  🎬  faststart aplicado: {trabajo['nombre_nuevo']}")
                except Exception as e:
                    log.warning(f"  faststart omitido en {trabajo['nombre_nuevo']}: {e}")
                    upload_path = tmp_path

            t_ul_ini = time.perf_counter()
            subir_a_blob_desde_path(
                blob_svc,
                upload_path,
                trabajo["blob_path"],
                content_type=content_type_para(trabajo["nombre_nuevo"]),
                max_concurrency=upload_concurrency,
            )
            t_ul = time.perf_counter() - t_ul_ini
            t_total = time.perf_counter() - t0

            size_mb = (trabajo.get("size_bytes", 0) or 0) / (1024 * 1024)
            dl_mbps = size_mb / t_dl if t_dl > 0 else 0.0
            ul_mbps = size_mb / t_ul if t_ul > 0 else 0.0
            eff_mbps = size_mb / t_total if t_total > 0 else 0.0

            return "ok", {
                "size_bytes": trabajo["size_bytes"],
                "dl_s": t_dl,
                "ul_s": t_ul,
                "total_s": t_total,
                "dl_mbps": dl_mbps,
                "ul_mbps": ul_mbps,
                "eff_mbps": eff_mbps,
            }
        except Exception as e:
            return "error", str(e)
        finally:
            for _p in (tmp_path, remux_path):
                if _p and os.path.exists(_p):
                    try:
                        os.remove(_p)
                    except OSError:
                        pass


def procesar_archivos(
    archivos: list[dict],
    blob_svc: BlobServiceClient,
    codigo_proyecto: str,
    nombre_corto: str,
    prefijo_jpg: str = "FOT",
    max_workers: int = 12,
) -> dict:
    """
    Procesa todos los archivos de un proyecto con estas mejoras:

    1. Construye el blob index una sola vez.
    2. Mantiene el mismo flujo de una sola ejecucion.
    3. Trata videos pesados con menos concurrencia y subida por archivo temporal.
    4. Trata el resto con mas paralelismo para no frenar el lote completo.
    """
    carpeta_proyecto = f"{codigo_proyecto}_{nombre_corto}"
    resumen = {
        "subidos": 0,
        "omitidos": 0,
        "errores": 0,
        "bytes_subidos": 0,
        "archivos_medidos": 0,
        "dl_mbps_prom": 0.0,
        "ul_mbps_prom": 0.0,
        "eff_mbps_prom": 0.0,
    }

    blob_index = construir_blob_index(blob_svc, prefix=carpeta_proyecto)
    trabajos = _preparar_trabajos(archivos, carpeta_proyecto, prefijo_jpg)

    if not trabajos:
        log.info(f"Sin archivos para procesar en {carpeta_proyecto}")
        return resumen

    nuevos = [t for t in trabajos if t["blob_path"] not in blob_index]
    ya_existentes = [t for t in trabajos if t["blob_path"] in blob_index]

    resumen["omitidos"] = len(ya_existentes)
    if ya_existentes:
        log.info(f"  ⏭️  {len(ya_existentes)} archivos ya existen en Blob, omitidos.")

    if not nuevos:
        log.info(f"  ✅  Nada nuevo que subir en {carpeta_proyecto}")
        return resumen

    pesados = [t for t in nuevos if es_trabajo_pesado(t)]
    livianos = [t for t in nuevos if t not in pesados]

    log.info(f"\n📁  {carpeta_proyecto}: {len(nuevos)} archivos nuevos a subir")
    if livianos:
        log.info(f"  • {len(livianos)} archivos livianos")
    if pesados:
        log.info(f"  • {len(pesados)} videos/archivos pesados")

    semaforo = threading.Semaphore(GRAPH_MAX_CONCURRENT)
    token = get_graph_token()
    dl_mbps_sum = 0.0
    ul_mbps_sum = 0.0
    eff_mbps_sum = 0.0

    grupos = [
        ("livianos", livianos, max(1, min(max_workers, len(livianos))), DOWNLOAD_CHUNK_SMALL, LIGHT_UPLOAD_CONCURRENCY),
        ("pesados", pesados, max(1, min(HEAVY_MAX_WORKERS, max_workers, len(pesados))), GRAPH_DOWNLOAD_CHUNK, VIDEO_UPLOAD_CONCURRENCY),
    ]

    for etiqueta, trabajos_grupo, workers, chunk_size, upload_concurrency in grupos:
        if not trabajos_grupo:
            continue
        log.info(f"  ▶ Procesando {etiqueta} con {workers} hilos")
        with ThreadPoolExecutor(max_workers=workers, thread_name_prefix=f"ripcon-{etiqueta}") as executor:
            futures = {
                executor.submit(
                    _procesar_un_trabajo,
                    trabajo,
                    blob_svc,
                    semaforo,
                    token,
                    chunk_size,
                    upload_concurrency,
                ): trabajo
                for trabajo in trabajos_grupo
            }
            for future in as_completed(futures):
                estado, payload = future.result()
                if estado == "ok":
                    resumen["subidos"] += 1
                    payload_ok = payload if isinstance(payload, dict) else {}
                    resumen["bytes_subidos"] += payload_ok.get("size_bytes", 0) or 0
                    dl_mbps_sum += payload_ok.get("dl_mbps", 0.0)
                    ul_mbps_sum += payload_ok.get("ul_mbps", 0.0)
                    eff_mbps_sum += payload_ok.get("eff_mbps", 0.0)
                    resumen["archivos_medidos"] += 1
                    log.info(
                        f"  📈  Throughput {futures[future]['nombre_nuevo']}: "
                        f"DL {payload_ok.get('dl_mbps', 0.0):.2f} MB/s | "
                        f"UL {payload_ok.get('ul_mbps', 0.0):.2f} MB/s | "
                        f"EFECTIVO {payload_ok.get('eff_mbps', 0.0):.2f} MB/s"
                    )
                    log.info(f"  ✅  Subido: {futures[future]['blob_path']}")
                else:
                    resumen["errores"] += 1
                    log.error(f"  ❌  Error con {futures[future]['nombre_orig']}: {payload}")


    if resumen["archivos_medidos"] > 0:
        n = resumen["archivos_medidos"]
        resumen["dl_mbps_prom"] = dl_mbps_sum / n
        resumen["ul_mbps_prom"] = ul_mbps_sum / n
        resumen["eff_mbps_prom"] = eff_mbps_sum / n
        log.info(
            f"  📊  Promedio throughput: DL {resumen['dl_mbps_prom']:.2f} MB/s | "
            f"UL {resumen['ul_mbps_prom']:.2f} MB/s | EFECTIVO {resumen['eff_mbps_prom']:.2f} MB/s"
        )

    return resumen


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MANTENIMIENTO: optimizar (faststart) los videos que YA estan en Azure
# ------------------------------------------------------------------------------
# Reescribe con el indice al frente los MP4/MOV que se subieron sin faststart,
# para que el hub los reproduzca al instante. Es sin perdida (copia streams).
#
# Como decide que tocar SIN descargar todo: lee unos pocos bytes del blob (rangos)
# y camina los "atomos". Solo baja/reescribe los que de verdad lo necesitan.
#
# Uso:
#   optimizar_videos_existentes(prefix="", dry_run=True)   # 1) ver que haria
#   optimizar_videos_existentes(prefix="", dry_run=False)  # 2) aplicarlo
#   prefix acota el barrido: "33014_promart-quitumbe" o "_media/videos", etc.
# ══════════════════════════════════════════════════════════════════════════════

def _blob_faststart_state(container_client, blob_name: str):
    """True = ya optimizado (moov antes que mdat); False = necesita faststart;
    None = no se pudo determinar. Solo lee unos bytes por rangos (no baja el archivo)."""
    bc = container_client.get_blob_client(blob_name)
    pos = 0
    try:
        for _ in range(32):
            header = bc.download_blob(offset=pos, length=16).readall()
            if len(header) < 8:
                return None
            size = int.from_bytes(header[:4], "big")
            box = header[4:8]
            if box == b"moov":
                return True
            if box == b"mdat":
                return False
            if size == 1:
                size = int.from_bytes(header[8:16], "big")
            elif size == 0:
                return None
            if size <= 0:
                return None
            pos += size
    except Exception:
        return None
    return None


def _download_blob_to_path(bc, destino_path: str) -> None:
    with open(destino_path, "wb") as f:
        bc.download_blob(max_concurrency=8).readinto(f)


def _tier_de(props) -> StandardBlobTier:
    try:
        return StandardBlobTier(props.blob_tier)
    except Exception:
        return StandardBlobTier.COOL


def _optimizar_un_blob_existente(container_client, blob_name: str, dry_run: bool) -> tuple[str, str]:
    estado = _blob_faststart_state(container_client, blob_name)
    if estado is not False:                 # True (ya ok) o None (no MP4/indeterminado)
        return ("ya_ok" if estado is True else "sin_dato"), blob_name
    if dry_run:
        return "por_optimizar", blob_name

    bc = container_client.get_blob_client(blob_name)
    suffix = Path(blob_name).suffix or ".mp4"
    tmp_in = tmp_out = None
    try:
        props = bc.get_blob_properties()
        ct = (props.content_settings.content_type if props.content_settings else None) or content_type_para(blob_name)
        tier = _tier_de(props)

        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tf:
            tmp_in = tf.name
        _download_blob_to_path(bc, tmp_in)

        if _mp4_faststart_state(tmp_in) is not False:   # recheck local
            return "ya_ok", blob_name

        tmp_out = _remux_faststart(tmp_in, suffix)
        with open(tmp_out, "rb") as data:
            container_client.upload_blob(
                name=blob_name,
                data=data,
                overwrite=True,
                standard_blob_tier=tier,
                max_concurrency=8,
                content_settings=ContentSettings(content_type=ct),
            )
        return "optimizado", blob_name
    finally:
        for _p in (tmp_in, tmp_out):
            if _p and os.path.exists(_p):
                try:
                    os.remove(_p)
                except OSError:
                    pass


def optimizar_videos_existentes(prefix: str = "", dry_run: bool = True, max_workers: int = 4) -> dict:
    if not FASTSTART_ENABLED:
        log.error("ffmpeg no esta instalado; no se puede optimizar. Instala ffmpeg y reintenta.")
        return {}

    blob_svc = get_blob_client()
    container = blob_svc.get_container_client(CONTAINER_NAME)
    candidatos = [
        b.name for b in container.list_blobs(name_starts_with=prefix)
        if Path(b.name).suffix.lower() in FASTSTART_EXTS
    ]
    log.info(f"Barriendo prefijo '{prefix or '(todo)'}': {len(candidatos)} MP4/MOV encontrados.")

    resumen = {"total": len(candidatos), "optimizados": 0, "ya_ok": 0, "por_optimizar": 0, "errores": 0}

    with ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix="fs-opt") as ex:
        futs = {ex.submit(_optimizar_un_blob_existente, container, name, dry_run): name for name in candidatos}
        for fut in as_completed(futs):
            name = futs[fut]
            try:
                estado, _ = fut.result()
            except Exception as e:
                resumen["errores"] += 1
                log.error(f"  ❌  {name}: {e}")
                continue
            if estado == "optimizado":
                resumen["optimizados"] += 1
                log.info(f"  🎬  optimizado: {name}")
            elif estado == "por_optimizar":
                resumen["por_optimizar"] += 1
                log.info(f"  [pendiente] por optimizar (dry-run): {name}")
            elif estado == "ya_ok":
                resumen["ya_ok"] += 1

    if dry_run:
        log.info(f"DRY-RUN: {resumen['por_optimizar']} por optimizar, {resumen['ya_ok']} ya OK. "
                 f"Vuelve a llamar con dry_run=False para aplicarlo.")
    else:
        log.info(f"Listo: {resumen['optimizados']} optimizados, {resumen['ya_ok']} ya estaban OK, "
                 f"{resumen['errores']} errores.")
    return resumen


# ── Ejemplo: primero en seco (solo reporta), luego aplica ──
# optimizar_videos_existentes(prefix="", dry_run=True)      # 1) ver que haria
# optimizar_videos_existentes(prefix="", dry_run=False)     # 2) aplicarlo de verdad


In [ ]:
# ONE DRIVE - USUARIO
resumen = upload_from_onedrive(
    user_email="wperalta@ripconciv.com",
    folder_path="/CONSULTORES/CC-VIVANTINO/VUELOS DRON",
    codigo_proyecto="33013",
    nombre_corto="vivantino",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=12
)
print(resumen)

In [ ]:
# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/33014%20PROMART%20QUITUMBE%20FASE%20IV%20EPC/03.%20INFORMACION%20TECNICA%20QUITUMBE%20EPC/03.10%20FOTOS%20Y%20VIDEOS"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="33014",
    nombre_corto="promart-quitumbe",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=10,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/BibliotecaProyectos/Documentos compartidos/General/00.Proyectos/98.Video 2020/Attiva"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="00000",
    nombre_corto="attiva",
    prefijo_jpg="FOT",
    recursive=True,
    max_workers=10,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/BibliotecaProyectos/Documentos compartidos/General/00.Proyectos/98.Video 2020/Puente Daule"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="00000",
    nombre_corto="puente-daule",
    prefijo_jpg="FOT",
    recursive=True,
    max_workers=8,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:
# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/22012%20PAVIMENTO%20ZONA%20W%20PTO%20POSORJA/02.%20PLANIFICACION%20Y%20CONTROL/02.6%20FOTOS%20Y%20VIDEOS"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="22012",
    nombre_corto="pavimento-zona-wpto-posorja",
    prefijo_jpg="FOT",
    recursive=True,
    max_workers=8,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:
# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/19004%20VIAL%204%20%20AV%20DE%20LAS%20AMERICAS/02.%20PLANIFICACION%20Y%20CONTROL/02.7%20FOTOS%20Y%20VIDEOS"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="19004",
    nombre_corto="vial-4-av-las-americas",
    prefijo_jpg="FOT",
    recursive=True,
    max_workers=8,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/22006%20PROMART%20ORELLANA/02.%20PLAN.%20Y%20CNTRL%20PROMART/02.6%20FOTOS%20Y%20VIDEOS%20PROMART"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="22006",
    nombre_corto="promart-orellana",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=12,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/GestindeProyectos/Documentos%20compartidos/22005%20PTAP%20MANCHA%20GRANDE/02.%20PLANIFICACION%20Y%20CONTROL/02.6%20FOTOS%20Y%20VIDEOS"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="22005",
    nombre_corto="ptap-mancha-grande",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=8,
)

print("\nResumen subida SharePoint:")
print(resumen)

In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/BibliotecaProyectos/Documentos%20compartidos/Miguel%20Larrea/FOTOGRAI%CC%81AS%20-%20VIDEOS"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="27020",
    nombre_corto="promart-congregacion-epc",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=16,
)

print("\nResumen subida SharePoint:")
print(resumen)



In [ ]:

# SharePoint exacto -> subir a Azure en carpeta 
sharepoint_url = "https://ripconcivcialtda.sharepoint.com/sites/BibliotecaProyectos/Documentos%20compartidos/General/00.Proyectos/Sim%C3%B3nBolivar/33.Puente%20los%20pajaros"

config_sp = resolver_sharepoint_desde_url(sharepoint_url)

print("Configuracion detectada:")
print(f"- Sitio: {config_sp['site_name']} ({config_sp['site_rel_path']})")
print(f"- Site ID: {config_sp['site_id']}")
print(f"- Carpeta origen en SharePoint: {config_sp['folder_path']}")

resumen = upload_from_sharepoint(
    site_id=config_sp["site_id"],
    folder_path=config_sp["folder_path"],
    codigo_proyecto="00000",
    nombre_corto="puente-los-pajaros",
    prefijo_jpg="DRN",
    recursive=True,
    max_workers=16,
)

print("\nResumen subida SharePoint:")
print(resumen)